1. Veri setini yükleme
2. MLP yapısını tanımlama
3. Model parametrelerini başlatma
4. İleri yayılım (forward propagation)
5. Maliyet (cost) hesaplama
6. Geri yayılım (Backpropagation)
7. Parametre güncelleme Update Propagation
8. Tüm adımların entegrasyonu

## Veri setini yükleme

In [47]:
import pandas as pd

In [48]:
df = pd.read_csv("drug_consumption_balanced.csv")

In [49]:
df['Cannabis'] = df['Cannabis'].apply(lambda x: 0 if x == 'CL0' else 1)

df = df.drop(columns= ['ID'])
cols = [c for c in df.columns if c != 'Cannabis'] + ['Cannabis']
df = df[cols]


In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 826 entries, 0 to 825
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Age        826 non-null    float64
 1   Gender     826 non-null    float64
 2   Education  826 non-null    float64
 3   Country    826 non-null    float64
 4   Ethnicity  826 non-null    float64
 5   Nscore     826 non-null    float64
 6   Escore     826 non-null    float64
 7   Oscore     826 non-null    float64
 8   Ascore     826 non-null    float64
 9   Cscore     826 non-null    float64
 10  Impulsive  826 non-null    float64
 11  SS         826 non-null    float64
 12  Cannabis   826 non-null    int64  
dtypes: float64(12), int64(1)
memory usage: 84.0 KB


In [51]:
df.head(50)

,Age,Gender,Education,Country,Ethnicity,Nscore,Escore,Oscore,Ascore,Cscore,Impulsive,SS,Cannabis
0,-0.95197,0.48246,1.16365,0.96082,-0.31685,-0.24649,0.47617,0.88309,-0.15487,1.81175,0.52975,0.40148,1
1,0.49788,0.48246,-0.05921,0.21128,-0.31685,0.04257,0.47617,-0.84732,-0.76096,1.13407,0.52975,0.76540,1
2,-0.07854,-0.48246,-0.61113,-0.57009,-0.31685,-0.34799,0.63779,1.88511,-0.60633,0.25953,0.19268,0.76540,1
3,0.49788,0.48246,1.98437,0.96082,-0.31685,0.52135,-0.30033,1.65653,0.28783,1.13407,-1.37983,-2.07848,1
4,-0.95197,-0.48246,-0.61113,-0.57009,-0.31685,-0.24649,0.16767,-0.97631,0.59042,0.93949,0.88113,0.40148,1
5,-0.07854,-0.48246,-0.05921,-0.57009,-0.31685,1.23461,-0.57545,0.88309,-0.01729,-0.89891,0.88113,0.76540,1
6,0.49788,0.48246,1.16365,0.96082,-0.31685,-0.58016,0.47617,-0.58331,-0.30172,0.93949,-0.21712,-0.84637,1
7,-0.07854,0.48246,1.98437,-0.57009,-0.31685,-1.05308,0.80523,-0.31776,-0.15487,0.93949,-0.21712,-0.52593,1
8,-0.07854,0.48246,-0.61113,-0.57009,-0.31685,-0.24649,-0.43999,-1.55521,0.13136,0.58489,-1.37983,-1.54858,1
9,-0.07854,0.48246,1.98437,0.96082,-0.31685,-1.05308,1.45421,-0.84732,0.94156,0.58489,-0.71126,-0.52593,1


In [52]:
df.describe()

,Age,Gender,Education,Country,Ethnicity,Nscore,Escore,Oscore,Ascore,Cscore,Impulsive,SS,Cannabis
count,826.000000,826.000000,826.000000,826.000000,826.000000,826.000000,826.000000,826.000000,826.000000,826.000000,826.000000,826.000000,826.0
mean,0.168080,0.085277,0.057608,0.508167,-0.328278,-0.060029,0.027327,-0.197196,0.110081,0.194614,-0.186178,-0.258899,1.0
std,0.909124,0.475151,0.984273,0.659379,0.173798,0.985466,0.978178,0.973696,0.969876,0.972190,0.938166,0.972517,0.0
min,-0.951970,-0.482460,-2.435910,-0.570090,-1.107020,-2.756960,-2.728270,-3.273930,-3.005370,-2.728270,-2.555240,-2.078480,1.0
25%,-0.951970,-0.482460,-0.611130,-0.285190,-0.316850,-0.763195,-0.575450,-0.847320,-0.606330,-0.405810,-0.711260,-0.846370,1.0
50%,-0.078540,0.482460,-0.059210,0.960820,-0.316850,-0.051880,0.003320,-0.177790,0.131360,0.259530,-0.217120,-0.215750,1.0
75%,1.094490,0.482460,0.454680,0.960820,-0.316850,0.629670,0.637790,0.445850,0.760960,0.939490,0.529750,0.401480,1.0
max,2.591710,0.482460,1.984370,0.960820,0.126000,3.273930,2.859500,2.449040,2.756960,3.464360,2.901610,1.921730,1.0


In [53]:
df.isnull().sum()

,0
Age,0
Gender,0
Education,0
Country,0
Ethnicity,0
Nscore,0
Escore,0
Oscore,0
Ascore,0
Cscore,0


In [54]:
df = df.sample(frac = 1, random_state = 42).reset_index(drop= True)

In [55]:
df.head(50)

,Age,Gender,Education,Country,Ethnicity,Nscore,Escore,Oscore,Ascore,Cscore,Impulsive,SS,Cannabis
0,-0.07854,0.48246,1.16365,0.96082,-0.22166,0.31287,0.63779,-0.58331,0.13136,-0.14277,0.19268,-0.21575,1
1,-0.95197,-0.48246,-0.61113,-0.57009,0.12600,-0.46725,2.12700,1.88511,-0.60633,-1.25773,1.86203,1.92173,1
2,-0.95197,-0.48246,-0.61113,-0.28519,-0.31685,-1.32828,-0.30033,0.88309,1.11406,-1.01450,-0.71126,-0.21575,1
3,1.09449,0.48246,1.16365,0.96082,-0.31685,-1.05308,1.74091,0.58331,0.43852,2.04506,-0.21712,-0.52593,1
4,-0.95197,0.48246,-0.61113,-0.57009,-0.31685,0.62967,0.63779,0.44585,0.13136,-2.57309,1.86203,1.22470,1
5,2.59171,-0.48246,0.45468,0.96082,-0.31685,-1.86962,-0.15487,-0.58331,2.46262,0.41594,-1.37983,-2.07848,1
6,-0.95197,-0.48246,-0.05921,0.96082,-0.31685,-0.92104,-0.30033,-0.71727,0.59042,0.41594,-0.71126,-1.54858,1
7,-0.95197,-0.48246,-1.43719,-0.09765,-0.31685,-0.34799,0.80523,0.72330,0.59042,-0.89891,1.86203,1.22470,1
8,1.09449,0.48246,0.45468,0.96082,-0.31685,0.62967,-0.57545,0.72330,-0.15487,0.41594,0.52975,-0.84637,1
9,1.09449,-0.48246,0.45468,0.96082,-0.31685,1.02119,-1.50796,-0.71727,0.28783,-0.89891,-0.21712,0.76540,1


In [56]:
X, y = df.iloc[:, :-1], df.iloc[:, -1]

In [57]:
print("X type is " + str(type(X)))
print("X shape is " + str(X.shape))
print("y type is " + str(type(y)))
print("y shape is " + str(y.shape))

X type is <class 'pandas.core.frame.DataFrame'>
X shape is (826, 12)
y type is <class 'pandas.core.series.Series'>
y shape is (826,)


In [58]:
X = X.to_numpy()


In [59]:
y = y.to_numpy().reshape(-1, 1)

In [60]:
print("X type is " + str(type(X)))
print("X shape is " + str(X.shape))
print("y type is " + str(type(y)))
print("y shape is " + str(y.shape))

X type is <class 'numpy.ndarray'>
X shape is (826, 12)
y type is <class 'numpy.ndarray'>
y shape is (826, 1)


In [61]:
from sklearn.model_selection import train_test_split

In [62]:
train_df = pd.read_csv('/content/drug_train.csv')
test_df = pd.read_csv('/content/drug_test.csv')

X_train = train_df.drop(columns=['Cannabis']).to_numpy()
y_train = train_df['Cannabis'].to_numpy().reshape(-1, 1)

X_test = test_df.drop(columns=['Cannabis']).to_numpy()
y_test = test_df['Cannabis'].to_numpy().reshape(-1, 1)

In [63]:
import numpy as np

print("Train class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
print(dict(zip(unique, counts)))

print("Test class distribution:")
unique, counts = np.unique(y_test, return_counts=True)
print(dict(zip(unique, counts)))

Train class distribution:
{np.int64(0): np.int64(330), np.int64(1): np.int64(330)}
Test class distribution:
{np.int64(0): np.int64(83), np.int64(1): np.int64(83)}


In [64]:
print("X train shape is " + str(X_train.shape))
print("X test shape is " + str(X_test.shape))
print("y train shape is " + str(y_train.shape))
print("y test shape is " + str(y_test.shape))

X train shape is (660, 12)
X test shape is (166, 12)
y train shape is (660, 1)
y test shape is (166, 1)


## Model Mimarisi (Define Network Structure)

Input layer size is X.shape[1]
Hidden layer size is 5.
Output layer size is 1.

Örneğin; giriş katmanı boyutu X.shape[1], gizli katman nöron sayısı 5 ve çıkış katmanı 1 olabilir.

In [65]:
print("X.shape is " + str(X.shape))
print("X.shape[1] is " + str(X.shape[1]))

X.shape is (826, 12)
X.shape[1] is 12


## Modeli Başlatmak (Initialize Model Parameters)

In [66]:
def initialize_parameters(n_x, n_h1, n_h2, n_y=1):
    np.random.seed(42)
    W1 = np.random.randn(n_h1, n_x) * np.sqrt(2. / n_x)
    b1 = np.zeros((n_h1, 1))
    W2 = np.random.randn(n_h2, n_h1) * np.sqrt(2. / n_h1)
    b2 = np.zeros((n_h2, 1))
    W3 = np.random.randn(n_y, n_h2) * np.sqrt(2. / n_h2)
    b3 = np.zeros((n_y, 1))

    parameters = {
        "W1": W1, "b1": b1,
        "W2": W2, "b2": b2,
        "W3": W3, "b3": b3
    }
    return parameters

In [67]:
test_parameters = initialize_parameters(X.shape[1], 5, 1)

In [68]:
print("W1 = " + str(test_parameters["W1"]))
print("b1 = " + str(test_parameters["b1"]))
print("W2 = " + str(test_parameters["W2"]))
print("b2 = " + str(test_parameters["b2"]))
print("W3 = " + str(test_parameters["W3"]))
print("b3 = " + str(test_parameters["b3"]))

W1 = [[ 0.2027827  -0.05644616  0.26441774  0.62177434 -0.09559271 -0.09558601
   0.64471093  0.31330392 -0.19166212  0.22149921 -0.18918948 -0.19013338]
 [ 0.09878068 -0.78109339 -0.70419476 -0.22955292 -0.41348657  0.12829094
  -0.37069928 -0.57657057  0.5983486  -0.09217279  0.02756827 -0.58165101]
 [-0.22224332  0.04528396 -0.46989116  0.15337807 -0.24520972 -0.11908347
  -0.2456457   0.7561894  -0.00551022 -0.43180868  0.33580255 -0.49840733]
 [ 0.08526821 -0.80003198 -0.54222968  0.08036826  0.30147772  0.06996081
  -0.04721321 -0.12292507 -0.60360407 -0.29387517 -0.18805499  0.43156834]
 [ 0.14028158 -0.71975813  0.13230673 -0.15720918 -0.27635225  0.2497158
   0.42090379  0.38019352 -0.34260912 -0.12623542  0.13523773  0.39826463]]
b1 = [[0.]
 [0.]
 [0.]
 [0.]
 [0.]]
W2 = [[-0.3030564  -0.11742105 -0.69970767 -0.7565475   0.51388645]]
b2 = [[0.]]
W3 = [[1.91801304]]
b3 = [[0.]]


In [69]:
print("W1 = " + str(test_parameters["W1"].shape))
print("b1 = " + str(test_parameters["b1"].shape))
print("W2 = " + str(test_parameters["W2"].shape))
print("b2 = " + str(test_parameters["b2"].shape))
print("W3 = " + str(test_parameters["W3"].shape))
print("b3 = " + str(test_parameters["b3"].shape))
print("X_train = " + str(X_train.shape))

W1 = (5, 12)
b1 = (5, 1)
W2 = (1, 5)
b2 = (1, 1)
W3 = (1, 1)
b3 = (1, 1)
X_train = (660, 12)


## Forward Propagation

In [70]:
def forward_propagation(X, parameters):
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    W3 = parameters["W3"]
    b3 = parameters["b3"]

    # gizli katman 1
    Z1 = np.dot(W1, X.T) + b1
    A1 = np.maximum(0, Z1)

    # gizli katman 2
    Z2 = np.dot(W2, A1) + b2
    A2 = np.maximum(0, Z2)

    # çıkış katmanı
    Z3 = np.dot(W3, A2) + b3
    A3 = 1 / (1 + np.exp(-Z3))

    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2, "Z3": Z3, "A3": A3}
    return A3, cache


In [71]:
def sigmoid(Z):
    return 1 / (1 + np.exp(-1 * Z))

In [72]:
test_A3, test_cache = forward_propagation(X_train, test_parameters)

In [73]:
print("A3 test:" + str(test_A3))
print("A3 test shape:" + str(test_A3.shape))

A3 test:[[0.5        0.5        0.5        0.5        0.5        0.5
  0.5        0.5        0.5        0.5        0.5        0.5
  0.5        0.5        0.5        0.5        0.5        0.5
  0.5        0.5        0.5        0.5        0.5        0.5
  0.5        0.5        0.5        0.5        0.5        0.5
  0.5        0.5        0.5        0.65261659 0.5        0.5
  0.5        0.5        0.5        0.5        0.5        0.5
  0.5        0.5        0.5        0.5        0.5        0.5
  0.5        0.5        0.5        0.5        0.5        0.5
  0.5        0.5        0.5        0.5        0.5        0.5
  0.5        0.5        0.5        0.5        0.5        0.5
  0.5        0.5        0.5        0.5        0.5        0.5
  0.5        0.5        0.5        0.5        0.5        0.5
  0.5        0.5        0.5        0.5        0.5        0.5
  0.5        0.5        0.5        0.5        0.5        0.5
  0.5        0.5        0.5        0.5        0.5        0.5
  0.5        0.5

In [74]:
print("Z1 is:" + str(test_cache["Z1"]))
print("A1 is:" + str(test_cache["A1"]))
print("Z2 is:" + str(test_cache["Z2"]))
print("A2 is:" + str(test_cache["A2"]))
print("Z3 is:" + str(test_cache["Z3"]))
print("A3 is:" + str(test_cache["A3"]))

Z1 is:[[ 0.2642465  -1.55293961  1.1323356  ...  1.24818692 -0.38530402
   0.09411224]
 [ 1.29259898  2.69170413  0.54295221 ... -1.84143817  1.21832267
  -0.32551391]
 [-0.17951881  1.10027563 -1.42351879 ...  0.97769385 -0.36940204
  -0.01148575]
 [ 0.10651001  1.88306948 -1.12002022 ...  2.26294772  0.8834147
  -1.21474675]
 [ 0.35134741  0.0738283  -1.49366595 ...  1.63754344 -0.88126435
  -0.77040529]]
A1 is:[[0.2642465  0.         1.1323356  ... 1.24818692 0.         0.09411224]
 [1.29259898 2.69170413 0.54295221 ... 0.         1.21832267 0.        ]
 [0.         1.10027563 0.         ... 0.97769385 0.         0.        ]
 [0.10651001 1.88306948 0.         ... 2.26294772 0.8834147  0.        ]
 [0.35134741 0.0738283  0.         ... 1.63754344 0.         0.        ]]
Z2 is:[[-0.13188713 -2.47262616 -0.40691557 -0.3655988  -2.73762526 -1.62469887
  -2.3561837  -0.90941508 -0.27595157 -2.35647408 -0.73189468 -0.62034891
  -0.24448745 -0.61665544 -1.84819625 -0.07403413 -0.86819669 -

## Compute Cost

In [75]:
print("Y shape is " +str(y_train.shape))

Y shape is (660, 1)


In [76]:
def compute_cost(A3, Y):
    m = A3.shape[1]
    cost = - (np.dot(np.log(A3), Y) + np.dot(np.log(1 - A3), (1 - Y))) / m
    cost = float(np.squeeze(cost))
    return cost

In [77]:
test_cost = compute_cost(test_A3, y_train)

In [78]:
print("test_cost is " + str(test_cost))
print("test_cost type is " + str(type(test_cost)))

test_cost is 0.6900340344955644
test_cost type is <class 'float'>


## Backpropagation

In [79]:
print("A1 shape: " + str(test_cache["A1"].shape))
print("A2 shape: " + str(test_cache["A2"].shape))
print("A3 shape: " + str(test_cache["A3"].shape))
print("W1 shape: " + str(test_parameters["W1"].shape))
print("W2 shape: " + str(test_parameters["W2"].shape))
print("W3 shape: " + str(test_parameters["W3"].shape))
print("y train shape: " + str(y_train.shape))
print("X_train shape: " + str(X_train.shape))

A1 shape: (5, 660)
A2 shape: (1, 660)
A3 shape: (1, 660)
W1 shape: (5, 12)
W2 shape: (1, 5)
W3 shape: (1, 1)
y train shape: (660, 1)
X_train shape: (660, 12)


In [80]:
def backpropagation(X, Y, cache, parameters):
    m = X.shape[0]

    W2 = parameters["W2"]
    W3 = parameters["W3"]
    A1 = cache["A1"]
    A2 = cache["A2"]
    A3 = cache["A3"]

    # çıkış katmanı gradyanları
    dZ3 = A3 - Y.T
    dW3 = np.dot(dZ3, A2.T) / m
    db3 = np.sum(dZ3, axis=1, keepdims=True) / m

    # gizli katman 2 gradyanları
    dZ2 = np.dot(W3.T, dZ3) * np.where(A2 > 0, 1, 0)
    dW2 = np.dot(dZ2, A1.T) / m
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m

    # gizli katman 1 gradyanları
    dZ1 = np.dot(W2.T, dZ2) * np.where(A1 > 0, 1, 0)
    dW1 = np.dot(dZ1, X) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m

    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2, "dW3": dW3, "db3": db3}
    return grads

In [81]:
test_grads = backpropagation(X_train, y_train, test_cache, test_parameters)

In [82]:
print("dW1 shape is: " + str(test_grads["dW1"].shape))
print("dW2 shape is: " + str(test_grads["dW2"].shape))
print("dW3 shape is: " + str(test_grads["dW3"].shape))
print("db1 shape is: " + str(test_grads["db1"].shape))
print("db2 shape is: " + str(test_grads["db2"].shape))
print("db3 shape is: " + str(test_grads["db3"].shape))
print("b1 shape is: " + str(test_parameters["b1"].shape))

dW1 shape is: (5, 12)
dW2 shape is: (1, 5)
dW3 shape is: (1, 1)
db1 shape is: (5, 1)
db2 shape is: (1, 1)
db3 shape is: (1, 1)
b1 shape is: (5, 1)


In [83]:
def update_parameters(parameters, grads, learning_rate=0.05):
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    W3 = parameters["W3"]
    b3 = parameters["b3"]

    dW1 = grads["dW1"]
    db1 = grads["db1"]
    dW2 = grads["dW2"]
    db2 = grads["db2"]
    dW3 = grads["dW3"]
    db3 = grads["db3"]

    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W3 -= learning_rate * dW3
    b3 -= learning_rate * db3

    parameters = {
        "W1": W1, "b1": b1,
        "W2": W2, "b2": b2,
        "W3": W3, "b3": b3
    }

    return parameters

In [84]:
test_learned_parameters = update_parameters(test_parameters, test_grads)

In [85]:
test_learned_parameters["b3"].shape

(1, 1)

In [86]:
def nn_model(X, Y, n_x, n_h1, n_h2, n_y, n_steps=100, print_cost=True):
    parameters = initialize_parameters(n_x, n_h1, n_h2, n_y)

    for i in range(0, n_steps):
        A3, cache = forward_propagation(X, parameters)
        cost = compute_cost(A3, Y)
        grads = backpropagation(X, Y, cache, parameters)
        parameters = update_parameters(parameters, grads)

        if print_cost and i % 10 == 0:
            print("cost %i %f" %(i, cost))

    return parameters

In [87]:
X_train.shape

(660, 12)

In [88]:
parameters = nn_model(X_train, y_train, X_train.shape[1], n_h1=3, n_h2=3, n_y=1, n_steps=500)

cost 0 0.688567
cost 10 0.684308
cost 20 0.680855
cost 30 0.677784
cost 40 0.674801
cost 50 0.671950
cost 60 0.669028
cost 70 0.665974
cost 80 0.662928
cost 90 0.659882
cost 100 0.656571
cost 110 0.653160
cost 120 0.649651
cost 130 0.645987
cost 140 0.642235
cost 150 0.638409
cost 160 0.634491
cost 170 0.630601
cost 180 0.626602
cost 190 0.622604
cost 200 0.618600
cost 210 0.614591
cost 220 0.610602
cost 230 0.606644
cost 240 0.602696
cost 250 0.598716
cost 260 0.594788
cost 270 0.590861
cost 280 0.587250
cost 290 0.583816
cost 300 0.580456
cost 310 0.577294
cost 320 0.574289
cost 330 0.571525
cost 340 0.568942
cost 350 0.566466
cost 360 0.564047
cost 370 0.561693
cost 380 0.559389
cost 390 0.557156
cost 400 0.554993
cost 410 0.552915
cost 420 0.550910
cost 430 0.549028
cost 440 0.547262
cost 450 0.545503
cost 460 0.543775
cost 470 0.542145
cost 480 0.540589
cost 490 0.539075


In [89]:
def predict(parameters, X):
    A3, cache = forward_propagation(X, parameters)
    predicts = A3 > 0.5
    return predicts

In [90]:
predicts = predict(parameters, X_test)

In [91]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
y_true = y_test.flatten()
y_pred = predicts.flatten()
acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='binary')
recall = recall_score(y_true, y_pred, average='binary')
f1 = f1_score(y_true, y_pred, average='binary')
conf_matrix = confusion_matrix(y_true, y_pred)
print(f"Accuracy: {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print("\nConfusion Matrix:")
print(conf_matrix)
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

Accuracy: 0.8313
Precision: 0.8481
Recall: 0.8072
F1 Score: 0.8272

Confusion Matrix:
[[71 12]
 [16 67]]

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.86      0.84        83
           1       0.85      0.81      0.83        83

    accuracy                           0.83       166
   macro avg       0.83      0.83      0.83       166
weighted avg       0.83      0.83      0.83       166



In [92]:
parameters_n_h = [i for i in range(3, 11)]
parameters_n_steps = [i for i in range(100, 1100, 100)]

for n_h in parameters_n_h:
    for n_step in parameters_n_steps:
        parameters = nn_model(X_train, y_train, X_train.shape[1], n_h1=n_h, n_h2=n_h, n_y=1, n_steps=n_step, print_cost=False)
        predicts = predict(parameters, X_test)
        acc = accuracy_score(y_test.flatten(), predicts.flatten())
        print(f"n_h1: {n_h}, n_h2: {n_h}, n_step: {n_step}, acc: {acc}")

n_h1: 3, n_h2: 3, n_step: 100, acc: 0.7530120481927711
n_h1: 3, n_h2: 3, n_step: 200, acc: 0.7771084337349398
n_h1: 3, n_h2: 3, n_step: 300, acc: 0.8373493975903614
n_h1: 3, n_h2: 3, n_step: 400, acc: 0.8373493975903614
n_h1: 3, n_h2: 3, n_step: 500, acc: 0.8313253012048193
n_h1: 3, n_h2: 3, n_step: 600, acc: 0.8313253012048193
n_h1: 3, n_h2: 3, n_step: 700, acc: 0.8373493975903614
n_h1: 3, n_h2: 3, n_step: 800, acc: 0.8313253012048193
n_h1: 3, n_h2: 3, n_step: 900, acc: 0.8313253012048193
n_h1: 3, n_h2: 3, n_step: 1000, acc: 0.8313253012048193
n_h1: 4, n_h2: 4, n_step: 100, acc: 0.7650602409638554
n_h1: 4, n_h2: 4, n_step: 200, acc: 0.8012048192771084
n_h1: 4, n_h2: 4, n_step: 300, acc: 0.8192771084337349
n_h1: 4, n_h2: 4, n_step: 400, acc: 0.8132530120481928
n_h1: 4, n_h2: 4, n_step: 500, acc: 0.7951807228915663
n_h1: 4, n_h2: 4, n_step: 600, acc: 0.7951807228915663
n_h1: 4, n_h2: 4, n_step: 700, acc: 0.7951807228915663
n_h1: 4, n_h2: 4, n_step: 800, acc: 0.8132530120481928
n_h1: 4, 